## **Extracción de Características Acústicas (Acoustic Embeddings)**

En este notebook abordamos la fase de extracción de características para la modalidad de audio. A diferencia de la transcripción de texto (donde importa qué se dice), en el análisis acústico nos centramos en cómo se dice. Para capturar esta información, hemos seleccionado dos tipos de codificadores, siguiendo un flujo de trabajo similar al notebook de extracción de características visuales:

1. **Wav2Vec 2.0**.
    * En concreto, hemos seleccionado el modelo **Wav2Vec 2.0 Base**, desarrollado por Facebook AI, aprende representaciones contextuales directamente de la forma de onda del audio sin procesar. Basado en Transformers, utiliza mecanismos de auto-atención (self-attention) para capturar dependencias temporales complejas. Ha sido pre-entrenado en miles de horas de audio no etiquetado (dataset de **LibriSpeech** con 1000 horas donde se lee en voz alta en inglés), generando embeddings que encapsulan información fonética, prosódica y emocional de alta calidad. Es el estándar de facto (SOTA). Se ha seleccionado el modelo **Base** (768) en lugar de **Large** (1024) ya que, al igual que el vídeo, los audios en MELD e IEMOCAP son muy cortos, por tanto un vector de 1024 dimensiones hubiera memorizado el ruido de fondo o particularidades del locutor en lugar de características emocionales generales. La versión **Base** ofrece una representación más generalista y robusta. Este modelo se carga desde la librería de `Transformers` de **Hugging Face** y, al ser un modelo público y abierto (`facebook/wav2vec2-base`), no necesitamos realizar login ni necesitamos token HF. 

        - **SALIDA**: Vectores contextuales de **768** dimensiones.

2. **MFCCs + Energía (RMS) + ZCR**.
    * Como alternativa ligera, extraemos características espectrales y prosódicas utilizando la librería `librosa`. 
    Los **MFCCs** (**Mel-frequency cepstral coefficients**) son coeficientes que representan el timbre de la voz tal y como lo percibe el oído humano. Son el estándar histórico en el reconocimiento de emociones. La **Energía** (**RMS**) y **ZCR** (*Zero-Crossing Rate*) son indicadores directos de la intensidad sonora (volumen) y la velocidad/"rugosidad" del habla, variables que suelen aumentar en estados de estrés, tal y como hemos detectado tras el EDA (que las personas estresadas tienden a hablar más rápido).

        - **MFCCs**: Extraeremos **13** coeficientes. Se seleccionan los primeros 13 ya que modelan eficazmente la envolvente del tracto vocal, descartando el ruido de alta frecuencia.
        - **Energía** (**RMS**): La raíz cuadrática media (*Root Mean Square*) es un indicador directo de la intensidad sonora. Responde a la pregunta de: *¿El sujeto grita o susurra?*. Extraeremos **1 RMS**.
        - **ZCR**: La tasa de cruces por cero mide la frecuencia de cambio de signo en la señal. Responde a la pregunta de: *¿Es una voz limpia, o hay ruido/rugosidad?*. Extraeremos **1 ZCR**.
    
        - **SALIDA**: Vectores de **15** dimensiones (13+1+1).



In [2]:
!pip install -q transformers librosa

In [ ]:
# Carga previa de todas las librerías y paquetes necesarios

import os
import numpy as np
import pandas as pd
import random
import librosa
import torch
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
from tqdm import tqdm  # Para barra de progreso
import random
from transformers import logging as hf_logging

# Silencionamos los warnings:
hf_logging.set_verbosity_error()

# Configuración de GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

Dispositivo: cuda


In [3]:
BASE_DIR = os.path.expanduser('/workspace') #Directorio base
CSV_PATH = os.path.join(BASE_DIR, 'Multimodal_Stress_Dataset.csv') # CSV global unificado
LOCAL_DATA_ROOT = os.path.join(BASE_DIR, 'EMBEDDINGS_AUDIO') # Directorio donde se almacenarán los features acústicos
os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

# Cargamos del CSV Global
if os.path.exists(CSV_PATH):
    df_global = pd.read_csv(CSV_PATH)
    display(df_global.head()) 
else:
    print(f"No se encuentra el archivo en {CSV_PATH}")

,Utterance_ID,Dialogue_ID,video_path,audio_path,Transcription,duration,split,target_stress,dataset_origin
0,train_dia0_utt0,0,train_splits/dia0_utt0.mp4,MELD_Audio/train_dia0_utt0.wav,also I was the point person on my company's tr...,5.672333,train,0,MELD
1,train_dia0_utt1,0,train_splits/dia0_utt1.mp4,MELD_Audio/train_dia0_utt1.wav,You must've had your hands full.,1.501500,train,0,MELD
2,train_dia0_utt2,0,train_splits/dia0_utt2.mp4,MELD_Audio/train_dia0_utt2.wav,That I did. That I did.,2.919583,train,0,MELD
3,train_dia0_utt3,0,train_splits/dia0_utt3.mp4,MELD_Audio/train_dia0_utt3.wav,So let's talk a little bit about your duties.,2.752750,train,0,MELD
4,train_dia0_utt4,0,train_splits/dia0_utt4.mp4,MELD_Audio/train_dia0_utt4.wav,My duties? All right.,6.464792,train,0,MELD


---

#### **ESTRUCTURA NOTEBOOK Y EJECUCIÓN**:

Al igual que el notebook de extracción de características visuales, este también se ejecutará en el servidor NVIDIA DGX al cual hemos tenido acceso para la realización de este TFG, debido a la alta carga computacional que supone la extracción de características de los clips de audio de ambos datasets. 

Primero, se realiza un análisis inicial para decidir la ventana temporal y frecuencia de muestro previa a la extracción. A continuación, se cargan el modelo y las funciones del módulo `feature` de `librosa` adecuadas, y se ejecuta el bucle final lanzado en el servidor para la extracción de características. 

Finalmente, se cargan los *embeddings* resultantes para realizar una auditoría final de validación (**sanity check**).


---

### **Análisis Previo**


Por un lado, para la decidir el tamaño de la ventana temporal, recuperamos los resultados obtenidos en el EDA por cada dataset:

* **MELD**:
    - ESTADÍSTICAS DE AUDIO: 
        - Duración Mínima: 0.08 segundos
        - Duración Máxima: 41.04 segundos
        - Media: 3.13 segundos
        - Percentil 95%: 7.88 segundos
        - Percentil 99%: 11.64 segundos

* **IEMOCAP**:
    - ESTADÍSTICAS DE AUDIO: 
        - Duración Mínima: 0.58 segundos
        - Duración Máxima: 34.14 segundos
        - Media: 4.56 segundos
        - Percentil 95%: 10.80 segundos
        - Percentil 99%: 15.31 segundos

Tal y como se indicó al final del notebook del EDA Multimodal, de acuerdo al percentil 99 de MELD y el percentil 95 de IEMOCAP, fijamos una ventana temporal de **11 segundos** para el audio. Con esto, conseguimos capturar aproximadamente el 99% de los clips de audio en MELD completos sin recortar y más del 95% en IEMOCAP, siendo el equilibrio perfecto para evitar la sobrecarga computacional.



**NOTA**: **El truncado a la ventana temporal definida y el padding de los clips más cortos no se aplican en este notebook, sino en el módulo `dataset.py`, durante la carga de datos para el entrenamiento. Esto permite modificar la ventana temporal como hiperparámetro sin necesidad de re-extraer los embeddings. Por tanto, en este notebook se extraen los embeddings de la longitud completa de cada clip, pero se justifica el tamaño de ventana que posteriormente utilizaremos.** Además, para el ajuste de hiperparámetros, se probará con una ventana temporal menor (de **7 segundos**) para comparar el rendimiento alcanzado por el modelo cuando se limita mucho más el contexto acústico.


Por otro lado, en cuanto a la frecuencia de muestreo, el modelo preentrenado de **Wav2Vec 2.0** trabaja con una frecuencia fijada en **16kHz** (es decir, se toman 16000 muestras del sonido por segundo) y en canal **mono**. Por tanto, es necesario 
que los clips de audio cargados tengan esta frecuencia. En **MFCCs + RMS + ZCR** 
también se fija esto mismo para hacer la comparación justa. 

Para lograr esta frecuencia de muestreo y canal mono cuando cargamos los clips de audio, utilizamos la librería `librosa`. Con el parámetro `sr` en `librosa.load`, se le indica la frecuencia de muestreo a `16000` y automáticamente esta librería detecta si el audio original está a una frecuencia distinta, y lo resamplea adecuadamente antes de devolverlo. Además, `librosa.load` también cuenta con un párametro por defecto que es `mono=True`, para que, si lee un audio estéreo (dos canales), `librosa` automáticamente hace la media de ambos canales y te devuelve un audio mono (1 canal). 

Además, como de forma predeterminada **Wav2Vec 2.0** aplica un `hop_length` de 320 muestras. Esto es que, internamente, genera un vector de características cada 20ms (con un stride acumulado de 320 muestras), que para una señal de 16kHz, esto resulta en 16000/320 = 50 vectores de características por segundo. Debemos fijar este mismo valor de `hop_length` para **MFCCs + RMS + ZCR** para hacer las comparaciones justas. 


----

La estructura de directorios, tras la ejecución de este notebook, quedará de la siguiente forma:
* **`EMBEDDINGS_AUDIO`**:

    * **`WAV2VEC`**: 

        * **`MELD`**: Embeddings de tamaño $(T,768)$ obtenidos de **Wav2Vec 2.0** para el dataset de **MELD**.

        * **`IEMOCAP`**: Embeddings de tamaño $(32,2048)$ obtenidos de **Wav2Vec 2.0** para el dataset de **IEMOCAP**.

    
    * **`MFCC`**: 

        * **`MELD`**: Embeddings de tamaño $(T,15)$ obtenidos de **MFCCs + RMS + ZCR** para el dataset de **MELD**.

        * **`IEMOCAP`**: Embeddings de tamaño $(T,15)$ obtenidos de **MFCCs + RMS + ZCR** para el dataset de **IEMOCAP**.

---
## **Wav2Vec 2.0**

In [5]:
# --- 1. Se carga inicialmente el modelo WAV2VEC 2.0 ---

def extract_wav2vec(audio_path,feature_extractor,model):
    """
    Con el modelo Wav2Vec 2.0 Base cargado previamente, extrae embeddings de audio.
    Retorna embeddings de tamaño (Time_Steps, 768).
    Sampling Rate obligatorio: 16000 Hz.
    """
    try:
        # Cargamos audio con obligatoriamente 16kHz:
        y, sr = librosa.load(audio_path, sr=16000, mono = True) # Indicamos explícitamente que mono es True (aunque ya este valor está por defecto)
        
        # Aplicamos preprocesamiento:
        inputs = feature_extractor(y, sampling_rate=16000, return_tensors="pt", padding=True)
        input_values = inputs.input_values.to(device)
        
        with torch.no_grad():
            outputs = model(input_values)
            # outputs.last_hidden_state tiene forma (1, T, 768) -> quitamos el batch con squeeze
            hidden_states = outputs.last_hidden_state.squeeze(0).cpu().numpy()
            
        return hidden_states
    except Exception as e:
        return None


---
## **MFCCs + RMS + ZCR**

In [8]:
# --- 2.MFCCs + RMS + ZCR ---
def extract_mfccs(audio_path):
    """
    Carga el audio, extrae 13 MFCCs, RMS y ZCR, y los apila en una matriz de características.
    Retorna matriz de (Time_Steps, 15).
    Compuesta por: 13 MFCCs + 1 RMS + 1 ZCR.
    """
    try:
        y, sr = librosa.load(audio_path, sr=16000, mono = True) # Indicamos explícitamente que mono es True (aunque ya este valor está por defecto) 
        hop_length = 320 # Mismo valor que en Wav2Vec
        
        # -----------> 13 MFCCs (Timbre / Tracto Vocal)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=hop_length, n_fft=512) # n_fft = 512 ya que para extraer los MFCC, librosa por defecto define una ventana de 2048 muestras, pero en MELD detectamos que hay ventanas de 0,08 segundos (lo que equivale a 1280 muestras). Establecemos una ventana de 512 muestras que es mucho menor a 1280 para que capture los audios cortos también sin añadir padding 
        
        # --------->RMS (Energía)
        rms = librosa.feature.rms(y=y, hop_length=hop_length)
        
        # ----------> ZCR (Rugosidad/Ruido)
        zcr = librosa.feature.zero_crossing_rate(y=y, hop_length=hop_length)
        
        # Apilamos: (15, T) -> Transponemos a (T, 15)
        features = np.vstack([mfcc, rms, zcr]).T
        return features
    except Exception as e:
        return None

----
## ***Feed-Forward*. Extracción de Características**

In [9]:
# --- BUCLE DE EJECUCIÓN ---

# Cargamos el modelo Wav2Vec 2.0 Base una sola vez para usarlo en todo el proceso:
# Cargamos también el feature extractor previo para el preprocesamiento acústico necesario (normalización a waveform):
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")
wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device)
wav2vec_model.eval()

modelos = ['wav2vec', 'mfcc']
origenes = ['MELD', 'IEMOCAP']


for modelo in modelos:
    print(f"\n---------Extracción con modelo: {modelo.upper()}----------\n")
    for origen in origenes:
        df_filtrado = df_global[df_global['dataset_origin']==origen].copy()
        for index, row in tqdm(df_filtrado.iterrows(), total=len(df_filtrado), desc=f"Extrayendo {modelo.upper()} - {origen.upper()}"):
            # Limpieza de la variable por cada iteración:
            emb = None

            # 1. Definimos los nombres de salida
            fname = f"{row['dataset_origin']}_{row['Utterance_ID']}.npy".replace("/", "_")
            out_path = os.path.join(LOCAL_DATA_ROOT, modelo.upper(), origen.upper(), fname)
        
            # Chequeo si ya existe (para reanudar si se corta la ejecución):
            if os.path.exists(out_path):
                continue


            # 2. Localizamos el archivo de entrada:
            if origen == 'MELD':
                audio_path = os.path.join('data', row['audio_path'])
            else :
                audio_path = os.path.join('data', 'IEMOCAP_Audio', row['audio_path'])
    
            # 3. Procesar si existe el archivo
            if os.path.exists(audio_path):
                if modelo == 'wav2vec':
                    emb = extract_wav2vec(audio_path, feature_extractor, wav2vec_model)
                else : # MFCCs + RMS + ZCR
                    emb = extract_mfccs(audio_path)
            
            # 4. Guardamos:
            if emb is not None :
                os.makedirs(os.path.dirname(out_path), exist_ok=True) # Creamos la carpeta por si no existe
                np.save(out_path, emb)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 211/211 [00:00<00:00, 32951.01it/s]



---------Extracción con modelo: WAV2VEC----------



Extrayendo WAV2VEC - IEMOCAP: 100%|████████████████████████████████████████████████| 7515/7515 [00:54<00:00, 138.73it/s]



---------Extracción con modelo: MFCC----------



Extrayendo MFCC - IEMOCAP: 100%|███████████████████████████████████████████████████| 7515/7515 [00:34<00:00, 216.15it/s]


### **Sanity Check**

Verificamos finalmente para asegurar que tenemos todos los *embeddings* cargados correctamente (conteo), además de mostrar un ejemplo verificando ese tamaño variable ya mencionado.

In [10]:
path_w2v = os.path.join(LOCAL_DATA_ROOT, 'WAV2VEC')
path_mfcc = os.path.join(LOCAL_DATA_ROOT, 'MFCC')
datasets = ['MELD', 'IEMOCAP']
for ds in datasets:
    # Construimos rutas específicas por dataset
    ds_w2v_path = os.path.join(path_w2v, ds)
    ds_hc_path = os.path.join(path_mfcc, ds)
    
    # Listamos archivos si las carpetas existen
    files_ds_w2v = os.listdir(ds_w2v_path) if os.path.exists(ds_w2v_path) else []
    files_ds_hc = os.listdir(ds_hc_path) if os.path.exists(ds_hc_path) else []
    
    # Total esperado según df_global
    esperado_ds = len(df_global[df_global['dataset_origin'] == ds])
    
    print(f"\n--- DATASET: {ds} ---")
    print(f"Esperados en CSV: {esperado_ds}")
    print(f"Wav2Vec generados: {len(files_ds_w2v)} ({(len(files_ds_w2v)/esperado_ds)*100:.1f}%)")
    print(f"MFCCs+RMS+ZCR generados: {len(files_ds_hc)} ({(len(files_ds_hc)/esperado_ds)*100:.1f}%)")

    # Verificación de dimensiones de un ejemplo aleatorio por cada dataset
    if len(files_ds_w2v) > 0:
        sample = random.choice(files_ds_w2v)
        
        # Cargamos los datos
        data_w2v = np.load(os.path.join(ds_w2v_path, sample))
        data_hc = np.load(os.path.join(ds_hc_path, sample))
        
        print(f"--> Ejemplo aleatorio: {sample}")
        print(f"--> Wav2Vec Shape: {data_w2v.shape} (Esperado: T, 768)")
        print(f"--> MFCC Shape:    {data_hc.shape}  (Esperado: T, 15)")


--- DATASET: MELD ---
Esperados en CSV: 13704
Wav2Vec generados: 13704 (100.0%)
MFCCs+RMS+ZCR generados: 13704 (100.0%)
--> Ejemplo aleatorio: MELD_train_dia693_utt5.npy
--> Wav2Vec Shape: (143, 768) (Esperado: T, 768)
--> MFCC Shape:    (145, 15)  (Esperado: T, 15)

--- DATASET: IEMOCAP ---
Esperados en CSV: 7515
Wav2Vec generados: 7515 (100.0%)
MFCCs+RMS+ZCR generados: 7515 (100.0%)
--> Ejemplo aleatorio: IEMOCAP_Ses02F_script02_2_M000.npy
--> Wav2Vec Shape: (104, 768) (Esperado: T, 768)
--> MFCC Shape:    (105, 15)  (Esperado: T, 15)
